# Coursework 1

This notebook is intended to be used as a starting point for your experiments. The instructions can be found in the MLP2025_26_CW1_Spec.pdf (see Learn,  Assignment Submission, Coursework 1). The methods provided here are just helper functions. If you want more complex graphs such as side by side comparisons of different experiments you should learn more about matplotlib and implement them. Before each experiment remember to re-initialize neural network weights and reset the data providers so you get a properly initialized experiment. For each experiment try to keep most hyperparameters the same except the one under investigation so you can understand what the effects of each are.

## Training Boilerplate

Use the below code as a boilerplate to start your experiments. You can add more cells or change the code as you see fit.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import logging
import sys
import os
from pathlib import Path
BASE = Path("D:/mlpractical").resolve()
os.environ["MLP_DATA_DIR"] = str(BASE / "data")
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

from mlp.data_providers import MNISTDataProvider, EMNISTDataProvider
from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, CustomActivationLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser

In [5]:
%matplotlib inline
plt.style.use('ggplot')

def train_model_and_plot_stats(
        model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True):
    
    # As well as monitoring the error over training also monitor classification
    # accuracy i.e. proportion of most-probable predicted classes being equal to targets
    data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

    # Use the created objects to initialise a new Optimiser instance.
    optimiser = Optimiser(
        model, error, learning_rule, train_data, valid_data, data_monitors, notebook=notebook)

    # Run the optimiser for num_epochs epochs (full passes through the training set)
    # printing statistics every epoch.
    stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)

    # Plot the change in the validation and training set error over training.
    fig_1 = plt.figure(figsize=(8, 4))
    ax_1 = fig_1.add_subplot(111)
    for k in ['error(train)', 'error(valid)']:
        ax_1.plot(np.arange(1, stats.shape[0]) * stats_interval, stats[1:, keys[k]], label=k)
    ax_1.legend(loc=0)
    ax_1.set_xlabel('Epoch number')
    ax_1.set_ylabel('Error')

    # Plot the change in the validation and training set accuracy over training.
    fig_2 = plt.figure(figsize=(8, 4))
    ax_2 = fig_2.add_subplot(111)
    for k in ['acc(train)', 'acc(valid)']:
        ax_2.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]], label=k)
    ax_2.legend(loc=0)
    ax_2.set_xlabel('Epoch number')
    ax_2.set_xlabel('Accuracy')

    grad_plot, grad_ax = optimiser.plot_grad_flow()

    return stats, keys, run_time, fig_1, ax_1, fig_2, ax_2, grad_plot, grad_ax

In [4]:
# The below code will set up the data providers, random number
# generator and logger objects needed for training runs. As
# loading the data from file take a little while you generally
# will probably not want to reload the data providers on
# every training run. If you wish to reset their state you
# should instead use the .reset() method of the data providers.

# Seed a random number generator
seed = 111020
rng = np.random.RandomState(seed)
batch_size = 100
# Set up a logger object to print info about the training run to stdout
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

# Create data provider objects for the MNIST data set
train_data = EMNISTDataProvider('train', batch_size=batch_size, rng=rng)
valid_data = EMNISTDataProvider('valid', batch_size=batch_size, rng=rng)

KeysView(NpzFile 'D:\\mlpractical\\data\\emnist-train.npz' with keys: inputs, targets)
KeysView(NpzFile 'D:\\mlpractical\\data\\emnist-valid.npz' with keys: inputs, targets)


In [ ]:
# Table 1 and Figure 2
learning_rate = 9e-4
num_epochs = 100
stats_interval = 1
batch_size = 100

input_dim, output_dim = 784, 47
weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

def build_one_hidden_model(hidden_dim):
    """1-hidden-layer: Affine -> ReLU -> Affine"""
    return MultipleLayerModel([
        AffineLayer(input_dim, hidden_dim, weights_init, biases_init),
        ReluLayer(),
        AffineLayer(hidden_dim, output_dim, weights_init, biases_init)
    ])

def run_width(width):
    model = build_one_hidden_model(width)
    error = CrossEntropySoftmaxError()
    learning_rule = AdamLearningRule(learning_rate=learning_rate)
    stats, keys, run_time, fig_1, ax_1, fig_2, ax_2, grad_plot, grad_ax = train_model_and_plot_stats(
        model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=False
    )
    train_err_last = float(stats[-1, keys['error(train)']])
    valid_err_last = float(stats[-1, keys['error(valid)']])
    valid_acc_last = float(stats[-1, keys['acc(valid)']]) * 100.0  # 转百分比
    return {
        'width': width,
        'Val. Acc (%)': valid_acc_last,
        'Train Error': train_err_last,
        'Val. Error': valid_err_last,
        'stats': stats,
        'keys': keys
    }

width_list = [32, 64, 128]
results = [run_width(w) for w in width_list]

# —— 打印 Table 1 要填的三列 —— 
print("Table 1 (1-hidden-layer, ReLU, Adam lr=9e-4, batch=100, epochs=100)")
for r in results:
    print(f"{r['width']:>5}  |  {r['Val. Acc (%)']:.2f}  |  {r['Train Error']:.3f}  |  {r['Val. Error']:.3f}")

# —— 画 Figure 2：把三个宽度的曲线画在一起 —— 
# (a) accuracy by epoch
fig_acc = plt.figure(figsize=(8,4))
ax_acc = fig_acc.add_subplot(111)
for r in results:
    stats, keys = r['stats'], r['keys']
    ax_acc.plot(np.arange(1, stats.shape[0]) * stats_interval, stats[1:, keys['acc(train)']],
                label=f"train, width={r['width']}")
    ax_acc.plot(np.arange(1, stats.shape[0]) * stats_interval, stats[1:, keys['acc(valid)']],
                label=f"valid, width={r['width']}", linestyle='--')
ax_acc.set_xlabel('Epoch number')
ax_acc.set_ylabel('Accuracy')
ax_acc.legend(loc=0)
ax_acc.set_title('Accuracy by epoch for different widths (1 hidden layer)')
fig_acc.tight_layout()
# fig_acc.savefig('figure2a_width_accuracy.pdf')  # 提交用矢量图可保存为PDF

# (b) error by epoch
fig_err = plt.figure(figsize=(8,4))
ax_err = fig_err.add_subplot(111)
for r in results:
    stats, keys = r['stats'], r['keys']
    ax_err.plot(np.arange(1, stats.shape[0]) * stats_interval, stats[1:, keys['error(train)']],
                label=f"train, width={r['width']}")
    ax_err.plot(np.arange(1, stats.shape[0]) * stats_interval, stats[1:, keys['error(valid)']],
                label=f"valid, width={r['width']}", linestyle='--')
ax_err.set_xlabel('Epoch number')
ax_err.set_ylabel('Error')
ax_err.legend(loc=0)
ax_err.set_title('Error by epoch for different widths (1 hidden layer)')
fig_err.tight_layout()
# fig_err.savefig('figure2b_width_error.pdf')  


In [ ]:
# Table 2 and Figure 3
learning_rate = 9e-4
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init  = ConstantInit(0.)

def build_model_with_depth(n_hidden_layers: int):
    layers = []
    # 首层：784 -> 128
    layers += [AffineLayer(input_dim, hidden_dim, weights_init, biases_init), ReluLayer()]
    # 中间的隐藏层（共有 n_hidden_layers - 1 个 128->128）
    for _ in range(n_hidden_layers - 1):
        layers += [AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), ReluLayer()]
    # 输出层：128 -> 47
    layers += [AffineLayer(hidden_dim, output_dim, weights_init, biases_init)]
    return MultipleLayerModel(layers)   

def run_depth(depth):
    model = build_model_with_depth(depth)
    error = CrossEntropySoftmaxError()
    learning_rule = AdamLearningRule(learning_rate=learning_rate)

    stats, keys, run_time, fig_err, ax_err, fig_acc, ax_acc, grad_plot, grad_ax = \
        train_model_and_plot_stats(model, error, learning_rule,
                                   train_data, valid_data,
                                   num_epochs, stats_interval, notebook=False)

    # —— 取最后一个 epoch 的三项指标，写 Table 2 ——
    last = stats.shape[0]-1
    train_err_last = float(stats[last, keys['error(train)']])
    valid_err_last = float(stats[last, keys['error(valid)']])
    valid_acc_last = float(stats[last, keys['acc(valid)']]) * 100.0

    return {
        'depth': depth,
        'Val. Acc (%)': valid_acc_last,
        'Train Error': train_err_last,
        'Val. Error': valid_err_last,
        'stats': stats, 'keys': keys,
        'fig_acc': fig_acc, 'ax_acc': ax_acc,
        'fig_err': fig_err, 'ax_err': ax_err
    }

# 跑三次：1层/2层/3层
results = [run_depth(d) for d in [1, 2, 3]]

# —— 打印 Table 2 三列 —— 
print("Table 2 (depth=1/2/3, width=128, Adam lr=9e-4, batch=100, epochs=100)")
for r in results:
    print(f"{r['depth']:>2} | {r['Val. Acc (%)']:.1f} | {r['Train Error']:.3f} | {r['Val. Error']:.3f}")

# —— 生成 Figure 3 —— 
# 3a: accuracy
plt.figure(figsize=(8,4))
for r in results:
    stats, keys = r['stats'], r['keys']
    x = np.arange(1, stats.shape[0]) * stats_interval
    plt.plot(x, stats[1:, keys['acc(train)']], label=f"depth {r['depth']} (train)")
    plt.plot(x, stats[1:, keys['acc(valid)']], label=f"depth {r['depth']} (valid)")
plt.xlabel("Epoch number"); plt.ylabel("Accuracy"); plt.legend(loc=0); plt.tight_layout()
# 保存为矢量图（便于放进 LaTeX）
plt.savefig("figure3a_accuracy.pdf")  # 建议PDF/SVG，模板也建议savefig输出矢量图。:contentReference[oaicite:4]{index=4}

# 3b: error
plt.figure(figsize=(8,4))
for r in results:
    stats, keys = r['stats'], r['keys']
    x = np.arange(1, stats.shape[0]) * stats_interval
    plt.plot(x, stats[1:, keys['error(train)']], label=f"depth {r['depth']} (train)")
    plt.plot(x, stats[1:, keys['error(valid)']], label=f"depth {r['depth']} (valid)")
plt.xlabel("Epoch number"); plt.ylabel("Error"); plt.legend(loc=0); plt.tight_layout()
plt.savefig("figure3b_error.pdf")   # 同上保存为矢量图。:contentReference[oaicite:5]{index=5}

In [ ]:
# The model set up code below is provided as a starting point.
# You will probably want to add further code cells for the
# different experiments you run.

%pip install tqdm

# Setup hyperparameters
learning_rate = 0.001
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

# Create model with TWO hidden layers
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # first hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # second hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

# Remember to use notebook=False when you write a script to be run in a terminal
stats, keys, run_time, fig_1, ax_1, fig_2, ax_2, grad_plot, grad_ax = train_model_and_plot_stats(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)

## Problematic Training

The following experiments show a failed training run. Run the code and investigate the reasons.

In [ ]:
# Setup hyperparameters
learning_rate = 0.001
num_epochs = 5
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

# Create model with four hidden layer
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), 
    CustomActivationLayer(), 
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init),
    CustomActivationLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init),
    CustomActivationLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init),
    CustomActivationLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

# Remember to use notebook=False when you write a script to be run in a terminal
stats, keys, run_time, fig_1, ax_1, fig_2, ax_2, grad_plot, grad_ax = train_model_and_plot_stats(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)

In [ ]:
# You can then check the plots like this
fig_1 # The training error plot

In [ ]:
fig_2 # The training accuracy plot

In [ ]:
grad_plot # The gradient flow plot